##**In Silico Identification of Novel Compounds for Insomnia disease Step 2: Molecular filtering: unwanted substructures and PAINS removal for the ENAMINE library.**


##**Author: Maurizio Rafael Hernández Díaz. 2025-2026 Course**

After selecting a number of compounds based on the algortihms constructed in `Step2.-QSAR model` it is important we do not include certain substructures in the next steps as certain substructures can be unfavorable as presenting toxic or reactive characteristics such as toxicity or reactivity Brenk et al. (Chem. Med. Chem. (2008), 3, 435-44) listed unfavorable subtructures such as nitrogenous groups with mutagenic propierties, sulfattes and phosphates that result in unfavorable pharmacokinetic propierties amounf others. On the other hand we will handle and drop "PAINS" compounds that are compounds that based on  Baell et al. (J. Med. Chem. (2010), 53, 2719-2740) ,those are compounds that  often are classified as a hit even they are false positives.This notebook has the objective of removing those undesired structured and PAINS in order to advance for the next steps.

In [1]:
!pip install rdkit
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

Failed to patch pandas - PandasTools will have limited functionality


In [2]:
#df=pd.read_csv(goodf)
Enamine_dataset=pd.read_csv("../DATA/Enamine_curated_database.csv")

##**1.-PAINS REMOVAL**

The PAINS filter is already implemented in RDKit. Such pre-defined filters can be applied via the FilterCatalog class.

In [3]:
Enamine_dataset.drop(columns="Unnamed: 0")
Enamine_dataset.columns

Index(['Unnamed: 0', 'SMILES', 'Catalog ID', 'MW', 'MW (desalted)', 'ClogP',
       'logS', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'AnalogsFromREAL', 'mol',
       'QED'],
      dtype='str')

In [4]:
Enamine_dataset["mol"]=Enamine_dataset["SMILES"].apply(lambda x: Chem.MolFromSmiles(x))

In [5]:
#We inititialize the filter
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

In [6]:

matches = []
clean_indices = []

for index, row in tqdm(Enamine_dataset.iterrows(), total=Enamine_dataset.shape[0]):
    molecule = row['mol']

    if molecule is None:
        continue
    entry = catalog.GetFirstMatch(molecule)

    if entry is not None:
        matches.append({
            "catalog_id": row['Catalog ID'],
            "unwanted_substructure": entry.GetDescription().capitalize(),
        })
    else:
        clean_indices.append(index)

Enamine_dataset_clean = Enamine_dataset.loc[clean_indices].copy()


  0%|          | 0/45119 [00:00<?, ?it/s]

In [7]:
print(f"Number of compounds with PAINS: {len(Enamine_dataset)}")
print(f"Number of compounds without PAINS: {len(Enamine_dataset_clean)}")

Number of compounds with PAINS: 45119
Number of compounds without PAINS: 44652


##**2.-FILTERING UNWANTED SUBSTRUCTURES**


Some lists of unwanted substructures, like PAINS, are already implemented in RDKit. However, it is also possible to use an external list and get the substructure matches manually. Here, I used the list provided in the supporting information from Brenk et al. (Chem. Med. Chem. (2008), 3, 535-44).

In [8]:
unwanted_brenk_etal=pd.read_csv("../DATA/filtros_brenk_final.tsv", sep="\t")
unwanted_brenk_etal
unwanted_brenk_etal.rename(columns={"SMARTS":"smarts"},inplace=True)
unwanted_brenk_etal

,ID,Name,smarts
0,3,acyclic C=C-O,C=[C!r]O
1,4,acyl cyanide,N#CC(=O)
2,5,acyl hydrazine,C(=O)N[NH2]
3,6,aldehyde,[CH1](=O)
4,7,Aliphatic long chain,[R0;D2][R0;D2][R0;D2][R0;D2]
5,8,alkyl halide,"[CX4][Cl,Br,I]"
6,9,amidotetrazole,c1nnnn1C=O
7,10,aniline,c1cc([NH2])ccc1
8,11,azepane,[CH2R2]1N[CH2R2][CH2R2][CH2R2][CH2R2][CH2R2]1
9,12,Azido group,N=[N+]=[N-]


In [9]:
unwanted_brenk_etal["rdkit_molecule"] = unwanted_brenk_etal["smarts"].apply(Chem.MolFromSmarts)

In [10]:

matches = []
clean_indices = []


for index, row in tqdm(Enamine_dataset.iterrows(), total=Enamine_dataset.shape[0]):
    molecule = row['mol']

    if molecule is None:
        continue

    has_unwanted = False

    for _, substructure in unwanted_brenk_etal.iterrows():
        if molecule.HasSubstructMatch(substructure.rdkit_molecule):
            matches.append({
                "catalog_id": row['Catalog ID'],
                "rdkit_molecule": molecule,
                "smarts": substructure['smarts']
            })
            has_unwanted = True
            break

    if not has_unwanted:
        clean_indices.append(index)


df_matches = pd.DataFrame(matches)


Enamine_dataset_clean = Enamine_dataset.loc[clean_indices].copy()

print(f"Dropped molecules: {len(df_matches)}")
print(f"Final number of molecules: {len(Enamine_dataset_clean)}")

  0%|          | 0/45119 [00:00<?, ?it/s]

Dropped molecules: 1242
Final number of molecules: 43877


In [11]:
Enamine_dataset_clean.to_csv("../DATA/Enamine_dataset_clean.csv")